In [1]:
from cr_element import CRElement
from cr_gamestate import CRGameState
from cr_dataset import CRDataset
import matplotlib.pyplot as plt
import cv2
import os
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image
from io import BytesIO
import numpy as np



cr_dataset = CRDataset()

In [50]:
import io
import pyarrow.parquet as pq
from PIL import Image, ImageDraw

# Grid parameters
Y_OFFSET_TOP = 62
Y_OFFSET_BOTTOM = 7
X_OFFSET_LEFT = 0
X_OFFSET_RIGHT = 0
NUM_ROWS = 32
NUM_COLS = 18

# Open the image
img = Image.open("test.png")
width, height = img.size

# Calculate tile dimensions from offsets
grid_width = width - X_OFFSET_LEFT - X_OFFSET_RIGHT
grid_height = height - Y_OFFSET_TOP - Y_OFFSET_BOTTOM
TILE_WIDTH = grid_width / NUM_COLS
TILE_HEIGHT = grid_height / NUM_ROWS

print(f"=== IMAGE DIMENSIONS ===")
print(f"Image size: {width} x {height} pixels")

print(f"\n=== CALCULATED TILE SIZE ===")
print(f"Grid area: {grid_width} x {grid_height} pixels")
print(f"Tile width: {TILE_WIDTH:.2f} pixels ({grid_width} / {NUM_COLS})")
print(f"Tile height: {TILE_HEIGHT:.2f} pixels ({grid_height} / {NUM_ROWS})")

# Create a drawing context
draw = ImageDraw.Draw(img)

# Draw vertical lines (columns)
for col in range(NUM_COLS + 1):
    x = X_OFFSET_LEFT + col * TILE_WIDTH
    y_start = Y_OFFSET_TOP
    y_end = Y_OFFSET_TOP + NUM_ROWS * TILE_HEIGHT
    draw.line([(x, y_start), (x, y_end)], fill="red", width=1)

# Draw horizontal lines (rows)
for row in range(NUM_ROWS + 1):
    x_start = X_OFFSET_LEFT
    x_end = X_OFFSET_LEFT + NUM_COLS * TILE_WIDTH
    y = Y_OFFSET_TOP + row * TILE_HEIGHT
    draw.line([(x_start, y), (x_end, y)], fill="red", width=1)

print(f"\n=== GRID PARAMETERS ===")
print(f"Offsets: left={X_OFFSET_LEFT}, right={X_OFFSET_RIGHT}, top={Y_OFFSET_TOP}, bottom={Y_OFFSET_BOTTOM}")
print(f"Grid: {NUM_COLS} columns x {NUM_ROWS} rows")
print(f"Grid covers: x=[{X_OFFSET_LEFT}, {width - X_OFFSET_RIGHT}], y=[{Y_OFFSET_TOP}, {height - Y_OFFSET_BOTTOM}]")

# Show the image
img.show()

=== IMAGE DIMENSIONS ===
Image size: 428 x 683 pixels

=== CALCULATED TILE SIZE ===
Grid area: 428 x 614 pixels
Tile width: 23.78 pixels (428 / 18)
Tile height: 19.19 pixels (614 / 32)

=== GRID PARAMETERS ===
Offsets: left=0, right=0, top=62, bottom=7
Grid: 18 columns x 32 rows
Grid covers: x=[0, 428], y=[62, 676]


In [45]:
img_match = cv2.imread("C:\\Users\\0dps1\\Downloads\\test.png")
# cv2.imshow("img_match", img_match)
# cv2.waitKey(0)
img_replay = cv2.imread("preview (1).jpg")
real_cards = CRElement.cut_to_fit(image=img_match, box=CRElement.BoundingBox(origin=(43,10), size=(457,737)))
real_cards = cv2.resize(real_cards, (428,683), dst=None, fx=None, fy=None, interpolation=cv2.INTER_LINEAR)
replay_cards = CRElement.cut_to_fit(image=img_replay, box=CRElement.BoundingBox(origin=(57,137), size=(428,683)))
cv2.imwrite("cut_new.png", real_cards)
# print(real_cards.shape)
cv2.imshow("a", real_cards)
cv2.waitKey(0)
cv2.imshow("a", replay_cards)
cv2.waitKey(0)
# for i in range(4):
#     print(real_cards[i].shape)
#     resized_image = cv2.resize(real_cards[i], (66,81), dst=None, fx=None, fy=None, interpolation=cv2.INTER_LINEAR)
#     print(replay_cards[i].shape)



-1

In [ ]:
# TODO
# Get decklist of replay
# Get cards played
# Save cards played
# (Randomly?) Select images where no card is played

In [ ]:
from huggingface_hub import snapshot_download
repo_id = "chrisrca/clash-royale-tv-replays"
arenas=["arena_27", "arena_26"] 
with open('HUGGING_KEY.txt', 'r') as f:
    for arena in arenas:
        snapshot_download(repo_id=repo_id, allow_patterns=f"{arena}/*/frames.parquet", repo_type="dataset", token=f.read(), cache_dir="F:/Dataset", dry_run=False, max_workers=16,)

Fetching 132 files:   0%|          | 0/132 [00:00<?, ?it/s]

LocalProtocolError: Illegal header value b'Bearer '

In [3]:
def get_event_log(replay, deck):
    old_hand = [None, None, None, None]
    prev_elixir = 0
    queued_elixir_losses = 0
    last_true_update_frame = 0
    event_log = []

    for frame_num, frame in enumerate(replay):
        elixir = CRGameState.current_elixir(frame, cr_dataset._elixirs)
        if elixir is not None:
            if(elixir - prev_elixir < -0.8): queued_elixir_losses += 1
            prev_elixir = elixir

        my_hand = CRGameState.cards_in_hand(frame, deck, elixir)
        empties = [index for index, element in enumerate(my_hand) if element == "empty"]
        nones = [index for index, element in enumerate(my_hand) if element is None]
        new_played = 0
        if(len(empties) > 0):
            for empty_index in empties:
                if old_hand[empty_index] != "empty":
                    event_log.append((last_true_update_frame, old_hand[empty_index]))
                    new_played += 1
                    old_hand[empty_index] = "empty"
        
        for i, card in enumerate(my_hand):
            if len(nones) > 0 or len(empties) > 0:
                continue
            name = card
            if(old_hand[i] != name):
                if(queued_elixir_losses > 0 and old_hand[i] is not None and "gray_" + name != old_hand[i] and "gray_" + old_hand[i] != name):

                    # played.append(old_hand[i])
                    event_log.append((last_true_update_frame, old_hand[i]))
                    new_played += 1
                    # print(old_hand, my_hand)
                    # print(f"Played {old_hand[i]} replaced by {name}")
                old_hand[i] = name
            # cv2.putText(frame, name, (CRElement.card.origin[0] + i * (CRElement.card_space + CRElement.card.size[0]), CRElement.card.origin[1]- i * 8), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0) if not gray else (255, 255, 255), 2)
        # TODO if empty appears, there is likely occlusion. Report card that empty replaced as played, but wait to update old hand.
        queued_elixir_losses -= new_played
        if (queued_elixir_losses < 0): queued_elixir_losses = 0
        if(len(empties) + len(nones) == 0): last_true_update_frame = frame_num
        # cv2.putText(frame, str(queued_elixir_losses), (40, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 5)
        # for i, card in enumerate(played):
        #     if(card == "empty"): continue
        #     cv2.putText(frame, card, (0, (i + 1) * 30), cv2.FONT_HERSHEY_SIMPLEX, 2, (255,0,255), 2)

        # If we lost 1 card and elixir went down that much (or that much - 1 if played before elixir addition (how to deal with elixir generators?)),
        # Count it as played
        # Update old_hand
        
        # cv2.putText(frame, str(elixir), (CRElement.elixir.origin[0], CRElement.elixir.origin[1]), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        # cv2.putText(frame, str(frame_num), (CRElement.elixir.origin[0], CRElement.elixir.origin[1]+20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 2)
        # writer.write(frame)

    # writer.release()
    return event_log
    

In [5]:
# print(cr_dataset.manifest)
snapshot = "F:\\Dataset\\datasets--chrisrca--clash-royale-tv-replays\\snapshots\\e186a4a8b5037b01a5da0dbfffa3404f08a5362c"
arenas = os.listdir(snapshot)
replays = []
for arena in arenas:
    root = f"{snapshot}\\{arena}"
    for replay in os.listdir(root):
        replays.append(f"{root}\\{replay}\\frames.parquet")
        if(cr_dataset.get_replay_from_manifest(f"{arena}/{replay}")) is None:
            cr_dataset.manifest.append({"replay": f"{arena}/{replay}", "local_dir": f"{root}\\{replay}\\frames.parquet"})

with open("manifest.json", "w") as json_file:
    json.dump(cr_dataset.manifest, json_file, indent=4)

In [ ]:
# # replays\arena_31\f7db1ae0-7e87-4dd4-991a-d88ff88bf262\gray_minions-1139.png
# # replays\arena_31\efb06ad5-5a2a-45f6-8e72-6d557d1654d3\gray_mini_pekka-699-next.png
# # replays\arena_31\efb06ad5-5a2a-45f6-8e72-6d557d1654d3\gray_golem-607-next.png
# # replays\arena_31\e73c9c99-867d-45fe-b072-d79961965947\gray_fireball-565.png
# # replays\arena_31\e69bbe32-8712-475d-80fe-df783430726b\gray_zap-639.png
# # replays\arena_31\e73c9c99-867d-45fe-b072-d79961965947\gray_hog_rider-379.png
# # replays\arena_31\6939743b-c09c-49cd-bcc0-2039cd882968\gray-minions-1091.png
# # replays\arena_31\5eacd6a6-1169-46df-b5e5-4b10798678fb\skeletons-326.png
# # replays\arena_31\5eacd6a6-1169-46df-b5e5-4b10798678fb\arrows-382.png
# replay = "673b5782-6e5e-4c00-a924-2f83a3aaf9ad"

# images = os.listdir(f"replays/arena_31/{replay}")
# # Get deck from image set
# deck_names = list(set([image.split("-")[0] for image in images]))
# deck = {name: card for name, card in cr_dataset.cards.items() if name in deck_names}
# g_minions_b4 = cv2.imread("replays\\arena_31\\673b5782-6e5e-4c00-a924-2f83a3aaf9ad\\evo_lumberjack-384.png")
# g_minions_next = cv2.imread("replays\\arena_31\\673b5782-6e5e-4c00-a924-2f83a3aaf9ad\\evo_lumberjack-384-next.png")

# print(get_event_log([g_minions_b4, g_minions_next], deck))

['electro_spirit', 'gray_tornado', 'gray_ice_golem', None]
['electro_spirit', 'gray_tornado', 'gray_ice_golem', 'gray_ram_rider']
[]


In [ ]:
validated_replays = [item['replay'] for item in cr_dataset.manifest if "local_dir" in item.keys() and not ("cards_identified" in item.keys())]

for replay_name in validated_replays:
    metadata = cr_dataset.get_replay_from_manifest(replay_name)
    if 'card_playtime' in metadata.keys() and metadata['card_playtime'] == True:
        continue
    print(replay_name)
    path = metadata["local_dir"]
    replay, deck, known_cards = cr_dataset.load_replay(replay_name=replay_name, clear_huggingface=False, local_pq_path=path)
    # shutil.rmtree("C:\\Users\\0dps1\\.cache\\huggingface\\hub\\datasets--chrisrca--clash-royale-tv-replays\\blobs")
    # shutil.rmtree("C:\\Users\\0dps1\\.cache\\huggingface\\hub\\datasets--chrisrca--clash-royale-tv-replays\\snapshots")
    # h0, w0 = replay[0].shape[:2]
    # writer = cv2.VideoWriter(f"vid.mp4", 0, 10, (w0, h0), isColor=True)
    # for frame in replay:
    #     writer.write(frame)
    # writer.release()
    if(known_cards):
        cr_dataset.get_replay_from_manifest(replay_name)['cards_identified'] = True
        event_log = get_event_log(replay, deck)
        for frame, name in event_log:
            os.makedirs(f"replays/{replay_name}", exist_ok=True)
            cv2.imwrite(f"replays/{replay_name}/{name}-{frame}.png", replay[frame])
            cv2.imwrite(f"replays/{replay_name}/{name}-{frame}-next.png", replay[frame + 1])
        cr_dataset.get_replay_from_manifest(replay_name)['card_playtime'] = True
        with open("manifest.json", "w") as json_file:
            json.dump(cr_dataset.manifest, json_file, indent=4)
    else:
        cr_dataset.get_replay_from_manifest(replay_name)['cards_identified'] = False
        print(f"Replay {replay_name} may have a new card")
        with open("manifest.json", "w") as json_file:
            json.dump(cr_dataset.manifest, json_file, indent=4)
    # Check if replay in manifest and cards_identified
    # if len([item for item in cr_dataset.manifest if item['replay'] == replay_name]) > 0:
    #     continue

    # replay = cr_dataset.load_replay(replay_name=replay_name)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'replays/arena_21/c3185448-87d0-42a0-a978-620e719a51b2'

In [47]:
validated_arenas = [arena for arena in os.listdir("replays")]
replays = []
for arena in validated_arenas:
    print(arena)
    for replay in os.listdir(f"replays/{arena}"):
        replays.append(f"{arena}/{replay}")
    card_played_frames = {replay: list((int(string.split(".")[0].split("-")[-1]), string.split(".")[0].split("-")[0]) for string in os.listdir(f"replays/{replay}") if string.split(".")[0].split("-")[-1].isdigit()) for replay in replays}
    print(card_played_frames['arena_21/1af27cb9-6903-421e-bd34-9506b41f6fc0'])
    snapshot = "F:\\Dataset\\datasets--chrisrca--clash-royale-tv-replays\\snapshots\\e186a4a8b5037b01a5da0dbfffa3404f08a5362c"
    data = {
        "card": [],
        "hand": [],
        "elixir": [],
        "png_bytes": [],
        "arena": [],
        "replay": [],
        "frame": [],
        "offset": []
    }
    # None played
    for replay in card_played_frames.keys():
        print(replay)
        deck_names = list(set([item[1] for item in card_played_frames[replay]]))
        deck = {name: card for name, card in cr_dataset.cards.items() if name in deck_names}
        print(deck)
        # Load parquet
        table = pq.read_table(f"{snapshot}/{replay}/frames.parquet")
        # Extract frames
        last_frame = 0
        for frame, name in card_played_frames[replay]:
            none_frame = (last_frame + frame) // 2
            column = table["image"]
            image_bytes = column[none_frame].as_py()['bytes']
            image_uncut = cv2.cvtColor(np.array(Image.open(BytesIO(image_bytes))), cv2.COLOR_RGB2BGR)
            elixir = CRGameState.current_elixir(image_uncut, cr_dataset._elixirs)
            image = CRElement.cut_to_fit(image_uncut, CRElement.arena)
            is_success, buffer = cv2.imencode(".png", image)
            data["card"].append("None")
            data["elixir"].append(elixir)
            data["hand"].append([card for card in CRGameState.cards_in_hand(image_uncut, deck, elixir)])
            data["png_bytes"].append(buffer.tobytes())
            data["arena"].append(arena)
            data["replay"].append(replay.split("/")[1])
            data["frame"].append(frame)
            data["offset"].append(0)
    new_table = pa.table({
        "card": pa.array(data["card"], type=pa.string()),
        "hand": pa.array(data["hand"], pa.list_(pa.string())),

        "elixir": pa.array(data["elixir"], type=pa.float16()),
        "png_bytes": pa.array(data["png_bytes"], type=pa.binary()),
        "arena": pa.array(data["arena"], type=pa.string()),
        "replay": pa.array(data["replay"], type=pa.string()),
        "frame": pa.array(data["frame"], type=pa.int16()),
        "offset": pa.array(data["offset"], type=pa.int16()),
    })

    parquet_path = f"Nones_{arena}.parquet"
    pq.write_table(
        new_table,
        str(parquet_path),
        compression="zstd",
        use_dictionary=True,
        write_statistics=True
    )
    continue
    data = {
        "png_bytes": [],
        "arena": [],
        "replay": [],
        "frame": [],
        "offset": []
    }
    
    for offset in [-1, -2, -3, -4, -5, -6, -7, -8]:
        print(offset)
        for replay in card_played_frames.keys():
            # Load parquet
            table = pq.read_table(f"{snapshot}/{replay}/frames.parquet")
            # Extract frames
            last_frame = 0
            for frame, name in card_played_frames[replay]:
                if "gray" in name: continue
                i = last_frame + offset
                if(i < 0): continue
                column = table["image"]
                image_bytes = column[none_frame].as_py()['bytes']
                image = cv2.cvtColor(np.array(Image.open(BytesIO(image_bytes))), cv2.COLOR_RGB2BGR)
                image = CRElement.cut_to_fit(image, CRElement.arena)
                is_success, buffer = cv2.imencode(".png", image)
                data["png_bytes"].append(buffer.tobytes())
                data["arena"].append(arena)
                data["replay"].append(replay.split("/")[1])
                data["frame"].append(frame)
                data["offset"].append(offset)
                
        new_table = pa.table({
            "png_bytes": pa.array(data["png_bytes"], type=pa.binary()),
            "arena": pa.array(data["arena"], type=pa.string()),
            "replay": pa.array(data["replay"], type=pa.string()),
            "frame": pa.array(data["frame"], type=pa.int16()),
            "offset": pa.array(data["offset"], type=pa.int16()),
        })

        parquet_path = f"offest_{offset}_{arena}.parquet"
        pq.write_table(
            new_table,
            str(parquet_path),
            compression="zstd",
            use_dictionary=True,
            write_statistics=True
        )
        
        


arena_21
[(127, 'arrows'), (369, 'arrows'), (550, 'arrows'), (107, 'balloon'), (337, 'balloon'), (569, 'balloon'), (533, 'barbs'), (419, 'fireball'), (622, 'fireball'), (441, 'inferno_dragon'), (588, 'inferno_dragon'), (91, 'inferno_dragon'), (259, 'lava_hound'), (488, 'lava_hound'), (27, 'skeleton_dragons'), (311, 'skeleton_dragons'), (523, 'skeleton_dragons'), (22, 'tombstone'), (239, 'tombstone'), (437, 'tombstone'), (606, 'tombstone')]
arena_21/1af27cb9-6903-421e-bd34-9506b41f6fc0
{'arrows': <cr_element.CRElement.Card object at 0x0000023A3D51F230>, 'balloon': <cr_element.CRElement.Card object at 0x0000023A3D4BAB40>, 'barbs': <cr_element.CRElement.Card object at 0x0000023A3D71E810>, 'fireball': <cr_element.CRElement.Card object at 0x0000023A3D752E10>, 'inferno_dragon': <cr_element.CRElement.Card object at 0x0000023A3D753B90>, 'lava_hound': <cr_element.CRElement.Card object at 0x0000023A3D753CB0>, 'skeleton_dragons': <cr_element.CRElement.Card object at 0x0000023A3DB70CB0>, 'tombston

FileNotFoundError: F:/Dataset/datasets--chrisrca--clash-royale-tv-replays/snapshots/e186a4a8b5037b01a5da0dbfffa3404f08a5362c/arena_31/5eacd6a6-1169-46df-b5e5-4b10798678fb/frames.parquet

In [7]:
def create_pyarrow_table():
    arenas = os.listdir("replays")
    data = {
        "card": [],
        "hand": [],
        "elixir": [],
        "png_bytes": [],
        "x": [],
        "y": [],
        "arena": [],
        "replay": [],
        "frame": [],
        "offset": []
    }
    i = 0
    for arena in arenas:
        if (arena in ["arena_27", "arena_28", "arena_29", "arena_31"]): continue
        replays = os.listdir(f"replays/{arena}")
        for replay in replays:
            images = os.listdir(f"replays/{arena}/{replay}")
            # Get deck from image set
            deck_names = list(set([image.split("-")[0] for image in images]))
            deck = {name: card for name, card in cr_dataset.cards.items() if name in deck_names}
            print(deck)
            for image_path in images:
                image = cv2.imread(f"replays/{arena}/{replay}/{image_path}")
                has_card_been_played = "next" in image_path
                gray = "gray" in image_path
                if(gray): continue
                image_uncut = image
                image = CRElement.cut_to_fit(image, CRElement.arena)
                is_success, buffer = cv2.imencode(".png", image)
                if is_success:
                    elixir = CRGameState.current_elixir(image_uncut, cr_dataset._elixirs)
                    data["card"].append(image_path.split("-")[0])
                    data["hand"].append([card for card in CRGameState.cards_in_hand(image_uncut, deck, elixir)])
                    data["elixir"].append(elixir)
                    data["png_bytes"].append(buffer.tobytes())
                    data["x"].append(-1)
                    data["y"].append(-1)
                    data["arena"].append(arena)
                    data["replay"].append(replay)
                    data["frame"].append(int(image_path.split(".")[0].split("-")[1]))
                    data["offset"].append(1 if has_card_been_played else 0)
                    i += 1
                    if i % 100 == 0:
                        print(i)
    return pa.table({
        "card": pa.array(data["card"], type=pa.string()),
        "hand": pa.array(data["hand"], pa.list_(pa.string())),
        "elixir": pa.array(data["elixir"], type=pa.float16()),
        "png_bytes": pa.array(data["png_bytes"], type=pa.binary()),
        "x": pa.array(data["x"], type=pa.int16()),
        "y": pa.array(data["y"], type=pa.int16()),
        "arena": pa.array(data["arena"], type=pa.string()),
        "replay": pa.array(data["replay"], type=pa.string()),
        "frame": pa.array(data["frame"], type=pa.int16()),
        "offset": pa.array(data["offset"], type=pa.int16()),
    })
                

table = create_pyarrow_table()
parquet_path = "new_arenas.parquet"
pq.write_table(
    table,
    str(parquet_path),
    compression="zstd",
    use_dictionary=True,
    write_statistics=True
)

{'arrows': <cr_element.CRElement.Card object at 0x0000023A3D51F230>, 'balloon': <cr_element.CRElement.Card object at 0x0000023A3D4BAB40>, 'barbs': <cr_element.CRElement.Card object at 0x0000023A3D71E810>, 'fireball': <cr_element.CRElement.Card object at 0x0000023A3D752E10>, 'inferno_dragon': <cr_element.CRElement.Card object at 0x0000023A3D753B90>, 'lava_hound': <cr_element.CRElement.Card object at 0x0000023A3D753CB0>, 'skeleton_dragons': <cr_element.CRElement.Card object at 0x0000023A3DB70CB0>, 'tombstone': <cr_element.CRElement.Card object at 0x0000023A3DB71040>}
{'cannon': <cr_element.CRElement.Card object at 0x0000023A3D71C770>, 'goblin_giant': <cr_element.CRElement.Card object at 0x0000023A3D753590>, 'guards': <cr_element.CRElement.Card object at 0x0000023A3D753800>, 'wizard': <cr_element.CRElement.Card object at 0x0000023A3DB713A0>, 'zap': <cr_element.CRElement.Card object at 0x0000023A3DB714C0>}
{'arrows': <cr_element.CRElement.Card object at 0x0000023A3D51F230>, 'goblin_gang': 

In [29]:
parquet_path = "initial_training_hand_elixir.parquet"
parquet = pq.read_table(parquet_path)
image = parquet["png_bytes"]
replays = parquet["replay"].to_pylist()
frames = parquet["frame"].to_pylist()
played = parquet["card"].to_pylist()
hands = parquet["hand"].to_pylist()
# replay = [index for index, replay in enumerate(replays) if replay == "6939743b-c09c-49cd-bcc0-2039cd882968"]
# frame = [index for index, frame in enumerate(frames) if frame == 1091]
# index = [index for index in frame if index in replay][0]
incorrect = []
for i, card in enumerate(played):
    if card not in hands[i]:
        print(card, hands[i], replays[i], frames[i])
        incorrect.append(i)

print(len(incorrect))
# print(replay)
# print(frame)
# print(index)
# png_byte_list = image.to_pylist()
# dataset = [cv2.cvtColor(np.array(Image.open(BytesIO(img))), cv2.COLOR_RGB2BGR) for img in png_byte_list]
# print(parquet["hand"][index])
# print(parquet["elixir"][index])
# cv2.imshow("test", dataset[index])
# cv2.waitKey(0)

evo_lumberjack ['electro_spirit', 'gray_tornado', 'gray_ice_golem', None] 673b5782-6e5e-4c00-a924-2f83a3aaf9ad 384
ice_spirit ['log', None, 'gray_cannon', 'gray_rocket'] 6762ff7a-193f-405e-b503-28b33de25a16 200
ice_spirit ['log', None, 'goblins', 'gray_rocket'] 6762ff7a-193f-405e-b503-28b33de25a16 443
ice_spirit ['royal_giant', 'goblins', None, 'rocket'] 6762ff7a-193f-405e-b503-28b33de25a16 49
lightning ['arrows', 'goblin_gang', 'minions', None] 6939743b-c09c-49cd-bcc0-2039cd882968 1054
lightning ['gray_arrows', 'gray_minions', 'gray_minions', None] 6939743b-c09c-49cd-bcc0-2039cd882968 1091
lightning ['goblin_gang', 'arrows', 'minions', None] 6939743b-c09c-49cd-bcc0-2039cd882968 616
lightning ['gray_goblin_gang', 'gray_arrows', 'gray_ice_wizard', None] 6939743b-c09c-49cd-bcc0-2039cd882968 626
snowball [None, 'fireball', 'barb_barrel', 'goblin_drill'] 6a732d40-dea6-42fe-b59f-cdfc09a2c4ad 1095
snowball ['barb_barrel', None, 'gray_fireball', 'skeletons'] 6a732d40-dea6-42fe-b59f-cdfc09a2c4

In [ ]:
# clock = cv2.imread("clock_2.png")
# gray = cv2.cvtColor(clock, cv2.COLOR_BGR2GRAY)
# mask = (gray > 5).astype(np.uint8) * 255
# th, tw = clock.shape[:2]

# # Add border around the source image so the template never clips
# padded_img = cv2.copyMakeBorder(
#     dataset[19],   # original image
#     top=th//2,
#     bottom=th//2,
#     left=tw//2,
#     right=tw//2,
#     borderType=cv2.BORDER_CONSTANT,
#     value=0        # black border (or whatever you like)
# )
# res = cv2.matchTemplate(padded_img, clock, cv2.TM_CCORR_NORMED, mask=mask)
# _, max_val, _, match_point = cv2.minMaxLoc(res)
# match_point = (match_point[0] + 34/2, match_point[1] + 40/2)
# print(max_val)    
# print(match_point)
# # Assume clock and dataset[18] are already loaded
# h, w = clock.shape[:2]  # height and width of template

# # Top-left corner from minMaxLoc
# top_left = (int(match_point[0] - w/2), int(match_point[1] - h/2))
# bottom_right = (top_left[0] + w, top_left[1] + h)

# # Draw rectangle on a copy of the image
# img_copy = padded_img.copy()
# cv2.rectangle(img_copy, top_left, bottom_right, color=(0, 255, 0), thickness=2)

# cv2.imshow("Matched Area", img_copy)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

0.8716230988502502
(130.0, 363.0)


In [ ]:
# def movement_highlight(backdrop: np.ndarray, keep_template: np.ndarray, replay: list[np.ndarray]):
#     arena_start = backdrop.copy()
#     gray = cv2.cvtColor(keep_template, cv2.COLOR_BGR2GRAY)
#     keep_mask = (gray < 5).astype(np.uint8) * 255
#     discard_mask = (gray > 250).astype(np.uint8) * 255
#     arena_start[discard_mask > 0] = 0

#     for image in replay:
#         snipped = image.copy()
#         snipped[discard_mask > 0] = 0

#         diff = cv2.absdiff(snipped, arena_start)  # per-pixel absolute difference
#         mask = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)  # collapse to grayscale if color
#         _, mask = cv2.threshold(mask, 35, 255, cv2.THRESH_BINARY)  # highlight significant differences
#         kernel_erode = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
#         kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

#         # Shrink (erode) then grow (dilate)
#         mask_clean = cv2.erode(mask, kernel_erode, iterations=1)
#         mask_clean = cv2.dilate(mask_clean, kernel_dilate, iterations=1)
#         print(np.sum(mask_clean > 0))

#         # Option 2: show only changed regions
#         changed_regions = cv2.bitwise_and(snipped, snipped, mask=mask_clean)

#         # manually draw over always-on regions
#         cv2.copyTo(image, keep_mask, changed_regions)
#         changed_regions[discard_mask > 0] = 0
#         cv2.imshow("Image", changed_regions)
#         cv2.waitKey(0)
#         cv2.destroyAllWindows()

# replay = [cv2.imread("test.png")]
# backdrop = cv2.imread("base.png")
# keep_template = cv2.imread("arena_31_template_2.png")
# movement_highlight(backdrop, keep_template, replay)

47086


In [ ]:
# arrows = cv2.imread("potential_new_cards\\b613634d-24f8-4e53-9feb-aef51e5c4a99.png")
# print(CRGameState.is_card_sliding(arrows))

# Incorrect replays


In [ ]:
# arrows = cv2.imread("potential_new_cards\\1dfcde88-9396-4fd3-9751-bf09159005c0.png")
# arrows_real = cv2.imread("cards\\evo_mega_knight-7.png")
# best_score, best_match = CRElement.best_match(
#     template=CRElement.prep(arrows), # TODO test different prepping downscaling for gray & non-gray (1,4)
#     labelled_image_collection={"arrows": CRElement.prep(arrows_real)},
#     shearing=(9,3)
# )
# print(best_score)